<a href="https://colab.research.google.com/github/eodenyire/LearnDataScienceWithEmmanuelOdenyire/blob/main/Phase_5_Specialization/Track_1_Data_Engineering/data_pipelines_with_kafka.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Pipelines with Apache Kafka
## Learning Objectives
- Understand Kafka architecture (brokers, topics, partitions)
- Build a producer-consumer pipeline
- Process streaming data in real time


In [ ]:
# Kafka concepts
kafka_concepts = {
    'Broker': 'A Kafka server that stores and serves messages',
    'Topic': 'A category or feed name for messages',
    'Partition': 'A topic is split into partitions for parallelism',
    'Producer': 'Publishes messages to topics',
    'Consumer': 'Reads messages from topics',
    'Consumer Group': 'Multiple consumers sharing a topic load',
    'Offset': 'Position of a message in a partition'
}
for k, v in kafka_concepts.items():
    print(f'  {k:20s}: {v}')

In [ ]:
# pip install confluent-kafka
# Simulated Kafka producer
class MockKafkaProducer:
    def __init__(self, config):
        self.config = config
        self.messages = []

    def produce(self, topic, key, value):
        self.messages.append({'topic': topic, 'key': key, 'value': value})
        print(f'Produced to {topic}: {value[:50]}...' if len(value) > 50 else f'Produced to {topic}: {value}')

    def flush(self):
        print(f'Flushed {len(self.messages)} messages')

producer = MockKafkaProducer({'bootstrap.servers': 'localhost:9092'})
for i in range(5):
    producer.produce('sensor-data', key=str(i), value=f'{{"sensor_id": {i}, "temp": {20+i}, "ts": "2024-01-0{i+1}"}}' )
producer.flush()

In [ ]:
# Simulated consumer pipeline
import json
from datetime import datetime

def process_sensor_event(event_str):
    event = json.loads(event_str)
    event['processed_at'] = datetime.now().isoformat()
    event['temp_fahrenheit'] = event['temp'] * 9/5 + 32
    return event

messages = ['{"sensor_id": 1, "temp": 22}', '{"sensor_id": 2, "temp": 35}']
for msg in messages:
    processed = process_sensor_event(msg)
    print(f'Processed: {processed}')

In [ ]:
# Kafka + Spark Streaming (conceptual)
spark_kafka_code = '''
from pyspark.sql import SparkSession
from pyspark.sql.functions import from_json, col

spark = SparkSession.builder.appName('KafkaSpark').getOrCreate()

df = spark.readStream \\
    .format('kafka') \\
    .option('kafka.bootstrap.servers', 'localhost:9092') \\
    .option('subscribe', 'sensor-data') \\
    .load()

# Parse JSON messages
parsed = df.select(from_json(col('value').cast('string'), schema).alias('data'))
query = parsed.writeStream.outputMode('append').format('console').start()
'''
print('Spark Structured Streaming + Kafka pattern:')
print(spark_kafka_code)

## Exercises
1. Set up a local Kafka cluster with Docker Compose.
2. Build a producer that reads CSV rows and streams them.
3. Build a consumer that aggregates streaming sensor data by minute.


## Summary
- Kafka enables fault-tolerant, high-throughput event streaming.
- Producers publish; consumers read from offsets.
- Spark Structured Streaming integrates with Kafka natively.
